<a href="https://colab.research.google.com/github/WVF-1/Movie-Success-Prediction-Model-Comparison/blob/main/Model%20Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 Predicting Movie Success — Notebook 11
## Model Training & Time-Series Cross-Validation

**Series:** May Newsletter — Movie Intelligence (Capstone)
**Prerequisite:** Run Notebook 10 first

### What we do here
1. Load the four feature matrices from NB10
2. Train one Random Forest classifier per pipeline using consistent hyperparameters
3. Evaluate each model with `TimeSeriesSplit` (5 folds, sorted by release year)
4. Record CV metrics: AUC-ROC, F1, Precision, Recall, MSE per fold
5. Save trained models and CV results for NB12

### Why TimeSeriesSplit?
Standard k-fold randomly shuffles folds, meaning a model trained on 2005 films can
validate on 1995 films — information flows backwards in time. `TimeSeriesSplit` ensures
each fold trains only on films *earlier* than the validation window, which is the honest
evaluation for a prediction task where "future" releases shouldn't inform the model.


## 0 · Setup

In [1]:
import pickle
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble         import RandomForestClassifier
from sklearn.model_selection  import TimeSeriesSplit, cross_validate
from sklearn.metrics          import (roc_auc_score, f1_score,
                                      precision_score, recall_score,
                                      mean_squared_error)
from sklearn.inspection       import permutation_importance

import os

RANDOM_STATE = 42
N_FOLDS      = 5

# Consistent RF hyperparameters across all four models for fair comparison
RF_PARAMS = dict(
    n_estimators  = 500,
    max_features  = "sqrt",
    min_samples_leaf = 5,
    class_weight  = "balanced",
    random_state  = RANDOM_STATE,
    n_jobs        = -1,
)

print("Scikit-learn RF parameters:")
for k, v in RF_PARAMS.items():
    print(f"  {k:<20}: {v}")


Scikit-learn RF parameters:
  n_estimators        : 500
  max_features        : sqrt
  min_samples_leaf    : 5
  class_weight        : balanced
  random_state        : 42
  n_jobs              : -1


## 1 · Load Feature Matrices

In [3]:
MODELS = ["M1_ROI","M2_Revenue","M3_Ratings","M4_CSI"]
LABELS = {
    "M1_ROI"     : "M1 — ROI",
    "M2_Revenue" : "M2 — Revenue",
    "M3_Ratings" : "M3 — Ratings",
    "M4_CSI"     : "M4 — CSI",
}

data = {}
for key in MODELS:
    with open(f"{key}.pkl","rb") as f:
        data[key] = pickle.load(f)
    d = data[key]
    print(f"  {key:<12} train={len(d['X_train']):,}  "
          f"test={len(d['X_test']):,}  features={len(d['feature_names'])}")


  M1_ROI       train=3,151  test=1,993  features=17
  M2_Revenue   train=3,151  test=1,993  features=20
  M3_Ratings   train=1,465  test=198  features=20
  M4_CSI       train=1,465  test=198  features=22


## 2 · TimeSeriesSplit Cross-Validation

In [4]:
tscv = TimeSeriesSplit(n_splits=N_FOLDS)

cv_results = {}

for key in MODELS:
    print(f"\n{'='*55}")
    print(f"  {LABELS[key]}")
    print(f"{'='*55}")

    X, y = data[key]["X_train"], data[key]["y_train"]
    model = RandomForestClassifier(**RF_PARAMS)

    fold_metrics = {m: [] for m in ["auc","f1","precision","recall","mse"]}

    for fold_idx, (train_idx, val_idx) in enumerate(tscv.split(X)):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        model.fit(X_tr, y_tr)
        y_prob = model.predict_proba(X_val)[:, 1]
        y_pred = model.predict(X_val)

        auc  = roc_auc_score(y_val, y_prob)
        f1   = f1_score(y_val, y_pred, zero_division=0)
        prec = precision_score(y_val, y_pred, zero_division=0)
        rec  = recall_score(y_val, y_pred, zero_division=0)
        mse  = mean_squared_error(y_val, y_prob)

        fold_metrics["auc"].append(auc)
        fold_metrics["f1"].append(f1)
        fold_metrics["precision"].append(prec)
        fold_metrics["recall"].append(rec)
        fold_metrics["mse"].append(mse)

        n_train_yrs = len(np.unique(
            data[key]["meta_train"].iloc[train_idx]["release_year"]))
        n_val_yrs   = len(np.unique(
            data[key]["meta_train"].iloc[val_idx]["release_year"]))

        print(f"  Fold {fold_idx+1}  "
              f"train_n={len(train_idx):,}  val_n={len(val_idx):,}  "
              f"|  AUC={auc:.3f}  F1={f1:.3f}  Prec={prec:.3f}  "
              f"Rec={rec:.3f}  MSE={mse:.4f}")

    cv_results[key] = {
        m: {"mean": np.mean(v), "std": np.std(v), "folds": v}
        for m, v in fold_metrics.items()
    }

    print(f"  {'─'*52}")
    for m in fold_metrics:
        r = cv_results[key][m]
        print(f"  {m:<12} mean={r['mean']:.4f}  std={r['std']:.4f}")



  M1 — ROI
  Fold 1  train_n=526  val_n=525  |  AUC=0.599  F1=0.608  Prec=0.624  Rec=0.594  MSE=0.2546
  Fold 2  train_n=1,051  val_n=525  |  AUC=0.619  F1=0.396  Prec=0.534  Rec=0.315  MSE=0.2466
  Fold 3  train_n=1,576  val_n=525  |  AUC=0.605  F1=0.503  Prec=0.538  Rec=0.473  MSE=0.2439
  Fold 4  train_n=2,101  val_n=525  |  AUC=0.573  F1=0.488  Prec=0.516  Rec=0.464  MSE=0.2545
  Fold 5  train_n=2,626  val_n=525  |  AUC=0.621  F1=0.505  Prec=0.541  Rec=0.473  MSE=0.2383
  ────────────────────────────────────────────────────
  auc          mean=0.6035  std=0.0172
  f1           mean=0.5001  std=0.0674
  precision    mean=0.5506  std=0.0376
  recall       mean=0.4634  std=0.0887
  mse          mean=0.2476  std=0.0063

  M2 — Revenue
  Fold 1  train_n=526  val_n=525  |  AUC=0.848  F1=0.652  Prec=0.557  Rec=0.785  MSE=0.1590
  Fold 2  train_n=1,051  val_n=525  |  AUC=0.875  F1=0.733  Prec=0.625  Rec=0.887  MSE=0.1609
  Fold 3  train_n=1,576  val_n=525  |  AUC=0.902  F1=0.826  Prec=0.7

## 3 · CV Summary Table

In [5]:
summary_rows = []
for key in MODELS:
    row = {"Model": LABELS[key]}
    for m in ["auc","f1","precision","recall","mse"]:
        r = cv_results[key][m]
        row[m.upper()] = f"{r['mean']:.3f} ± {r['std']:.3f}"
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index("Model")
print("\nCV Summary (mean ± std across 5 TimeSeriesSplit folds):")
print(summary_df.to_string())



CV Summary (mean ± std across 5 TimeSeriesSplit folds):
                        AUC             F1      PRECISION         RECALL            MSE
Model                                                                                  
M1 — ROI      0.603 ± 0.017  0.500 ± 0.067  0.551 ± 0.038  0.463 ± 0.089  0.248 ± 0.006
M2 — Revenue  0.884 ± 0.027  0.767 ± 0.067  0.704 ± 0.096  0.850 ± 0.042  0.142 ± 0.018
M3 — Ratings  0.851 ± 0.043  0.746 ± 0.032  0.750 ± 0.051  0.744 ± 0.020  0.155 ± 0.019
M4 — CSI      0.886 ± 0.018  0.801 ± 0.046  0.819 ± 0.073  0.785 ± 0.031  0.143 ± 0.014


## 4 · Train Final Models on Full Training Set

In [6]:
# Retrain each model on the complete training set before test-set evaluation in NB12
final_models = {}

for key in MODELS:
    X, y = data[key]["X_train"], data[key]["y_train"]
    model = RandomForestClassifier(**RF_PARAMS)
    model.fit(X, y)
    final_models[key] = model
    print(f"  {key:<12} trained on {len(X):,} films  "
          f"(OOB approx. accuracy: not computed — set oob_score=True to enable)")

print("\nAll four final models trained ✔")


  M1_ROI       trained on 3,151 films  (OOB approx. accuracy: not computed — set oob_score=True to enable)
  M2_Revenue   trained on 3,151 films  (OOB approx. accuracy: not computed — set oob_score=True to enable)
  M3_Ratings   trained on 1,465 films  (OOB approx. accuracy: not computed — set oob_score=True to enable)
  M4_CSI       trained on 1,465 films  (OOB approx. accuracy: not computed — set oob_score=True to enable)

All four final models trained ✔


## 5 · Permutation Feature Importance (Training Set)

In [7]:
# Use permutation importance rather than impurity-based (more honest for mixed types)
# Computed on a held-aside slice of the training set to keep it fast

perm_results = {}

for key in MODELS:
    X, y   = data[key]["X_train"], data[key]["y_train"]
    model  = final_models[key]
    names  = data[key]["feature_names"]

    # Use last 20% of training set as importance reference slice
    n_ref = max(200, int(len(X) * 0.20))
    X_ref, y_ref = X[-n_ref:], y[-n_ref:]

    perm = permutation_importance(
        model, X_ref, y_ref,
        n_repeats=10, random_state=RANDOM_STATE,
        scoring="roc_auc", n_jobs=-1
    )

    imp_df = pd.DataFrame({
        "feature"    : names,
        "importance" : perm.importances_mean,
        "std"        : perm.importances_std,
    }).sort_values("importance", ascending=False).reset_index(drop=True)

    perm_results[key] = imp_df

    print(f"\n{LABELS[key]} — Top 10 features:")
    print(imp_df.head(10)[["feature","importance","std"]].to_string(index=False))



M1 — ROI — Top 10 features:
      feature  importance      std
   log_budget    0.191949 0.013320
runtime_clean    0.154463 0.008239
    month_sin    0.093209 0.009792
    month_cos    0.074275 0.012309
  genre_count    0.068684 0.004570
  genre_Drama    0.035110 0.002640
 genre_Comedy    0.028356 0.005253
 genre_Horror    0.023865 0.003835
 genre_Action    0.023476 0.002787
   is_english    0.016745 0.001101

M2 — Revenue — Top 10 features:
               feature  importance      std
        has_known_lead    0.108892 0.007252
            log_budget    0.063788 0.005928
director_success_score    0.043516 0.003256
       cast_size_clean    0.021239 0.001890
         runtime_clean    0.014063 0.001179
             month_sin    0.010728 0.001618
             month_cos    0.008391 0.000582
           genre_count    0.006715 0.000517
           genre_Drama    0.004424 0.000620
          genre_Comedy    0.002474 0.000265

M3 — Ratings — Top 10 features:
               feature  importance  

## 6 · Save Models & Results

In [8]:
save_payload = {
    "final_models" : final_models,
    "cv_results"   : cv_results,
    "perm_results" : perm_results,
    "data"         : data,
    "LABELS"       : LABELS,
}

with open("capstone_models.pkl","wb") as f:
    pickle.dump(save_payload, f)

print("Saved: capstone_models.pkl")
print()
print("Contents:")
print("  final_models  — four trained RandomForestClassifiers")
print("  cv_results    — fold-level metrics for all four models")
print("  perm_results  — permutation importances for all four models")
print("  data          — feature matrices and metadata")
print()
print("Proceed to Notebook 12 → Evaluation & Newsletter Visuals ▶")


Saved: capstone_models.pkl

Contents:
  final_models  — four trained RandomForestClassifiers
  cv_results    — fold-level metrics for all four models
  perm_results  — permutation importances for all four models
  data          — feature matrices and metadata

Proceed to Notebook 12 → Evaluation & Newsletter Visuals ▶
